# 04 · Gold Layer test

**Transition tested:** Silver → Gold

| Suite | Category | Description |
|---|---|---|
| T1 | **Reconciliation** | Gold row counts vs Silver source of truth |
| T2 | **Row-to-Row Integrity** | Every Silver listing_id present in fact_listings |
| T3 | **Audit Columns** | `gold_load_dt` on all tables; `__START_AT/__END_AT` on SCD2 |
| T4 | **Schema Integrity** | All FK + derived + financial columns present |
| T5 | **Star Schema Joins** | FK join rates ≥ 60%, orphan check |
| T6 | **SCD2 Integrity** | One CURRENT row per key, END_AT consistency |

## Setup & Configuration

In [0]:
from pyspark.sql import functions as F

dbutils.widgets.text("project_catalog", "vstone_catalog")
dbutils.widgets.text("gold_schema",     "gold")
dbutils.widgets.text("silver_schema",   "silver")

CATALOG = dbutils.widgets.get("project_catalog")
GOLD    = f"{CATALOG}.{dbutils.widgets.get('gold_schema')}"
SILVER  = f"{CATALOG}.{dbutils.widgets.get('silver_schema')}"

MIN_JOIN_RATE_PCT = 60.0
DIM_DATE_EXPECTED = 7671

ALL_RESULTS = []
def record(suite, name, status, detail=""):
    ALL_RESULTS.append({"suite": suite, "test": name, "status": status, "detail": detail})
    tag = " PASS" if status == "PASS" else " FAIL"
    print(f"  {tag} | {suite} | {name:<45} | {detail}")

print(f"  GOLD QA SUITE  →  {GOLD}")
print(f"  SILVER SOURCE  →  {SILVER}")
print("=" * 70)

## T1 — Reconciliation (Silver → Gold)

In [0]:
# ══════════════════════════════════════════════════════════════════════════════
# T1 — RECONCILIATION  (Silver → Gold row counts)
# Rule: fact_listings must be 1:1 with listings_silver_merged.
#       SCD2 dims: CURRENT rows = __END_AT IS NULL (DLT does not add __CURRENT).
#       dim_date: exactly 7671 rows.
# ══════════════════════════════════════════════════════════════════════════════
print("\nT1 ── RECONCILIATION (Silver → Gold) ─────────────────────────────────")

# T1.1 fact_listings = listings_silver_merged (1:1)
silver_cnt = spark.table(f"{SILVER}.listings_silver_merged").count()
gold_cnt   = spark.table(f"{GOLD}.fact_listings").count()
diff_pct   = abs(gold_cnt - silver_cnt) / max(silver_cnt, 1) * 100
record("T1", "fact_listings row count matches Silver (1:1)",
       "PASS" if diff_pct <= 1.0 else "FAIL",
       f"Silver={silver_cnt:,}  Gold={gold_cnt:,}  Diff={diff_pct:.3f}%")

# T1.2 dim_car CURRENT = DISTINCT brand+model in Silver
# Current row definition: __END_AT IS NULL (DLT SCD2 has no __CURRENT column)
silver_car = spark.table(f"{SILVER}.car_catalog_transformation").select("brand", "model").distinct().count()
gold_car   = spark.table(f"{GOLD}.dim_car").filter(F.col("__END_AT").isNull()).count()
record("T1", "dim_car CURRENT (__END_AT IS NULL) = DISTINCT brand+model in Silver",
       "PASS" if gold_car == silver_car else "FAIL",
       f"Silver DISTINCT={silver_car:,}  dim_car CURRENT={gold_car:,}")

# T1.3 dim_location CURRENT = DISTINCT city_prepositional in Silver
silver_loc = spark.table(f"{SILVER}.geography_transformation").select("city_prepositional").distinct().count()
gold_loc   = spark.table(f"{GOLD}.dim_location").filter(F.col("__END_AT").isNull()).count()
record("T1", "dim_location CURRENT (__END_AT IS NULL) = DISTINCT city_prepositional in Silver",
       "PASS" if gold_loc == silver_loc else "FAIL",
       f"Silver DISTINCT={silver_loc:,}  dim_location CURRENT={gold_loc:,}")

# T1.4 dim_listing_details CURRENT = DISTINCT listing_id in Silver
silver_det = spark.table(f"{SILVER}.listings_text_transformation").select("listing_id").distinct().count()
gold_det   = spark.table(f"{GOLD}.dim_listing_details").filter(F.col("__END_AT").isNull()).count()
record("T1", "dim_listing_details CURRENT (__END_AT IS NULL) = DISTINCT listing_id in Silver",
       "PASS" if gold_det == silver_det else "FAIL",
       f"Silver DISTINCT={silver_det:,}  dim_listing_details CURRENT={gold_det:,}")

# T1.5 dim_listing_photos CURRENT = DISTINCT listing_id+photo_url_clean in Silver
silver_photo = spark.table(f"{SILVER}.listings_photo_transformation").select("listing_id", "photo_url_clean").distinct().count()
gold_photo   = spark.table(f"{GOLD}.dim_listing_photos").filter(F.col("__END_AT").isNull()).count()
record("T1", "dim_listing_photos CURRENT (__END_AT IS NULL) = DISTINCT listing_id+photo_url_clean",
       "PASS" if gold_photo == silver_photo else "FAIL",
       f"Silver DISTINCT={silver_photo:,}  dim_listing_photos CURRENT={gold_photo:,}")

# T1.6 dim_date = 7671
date_cnt = spark.table(f"{GOLD}.dim_date").count()
record("T1", f"dim_date = {DIM_DATE_EXPECTED:,} rows (2010-2030)",
       "PASS" if date_cnt == DIM_DATE_EXPECTED else "FAIL",
       f"Actual={date_cnt:,}  Expected={DIM_DATE_EXPECTED:,}")

# T1.7 All 5 aggregates populated
for agg in ["agg_monthly_sales_trend", "agg_brand_location_performance",
            "agg_regional_market_depth", "agg_comprehensive_kpi_cube", "agg_top_10_brands_by_spend"]:
    cnt = spark.table(f"{GOLD}.{agg}").count()
    record("T1", f"{agg} is populated", "PASS" if cnt > 0 else "FAIL", f"Rows={cnt:,}")

## T2 — Row-to-Row Integrity

In [0]:
from pyspark.sql import functions as F

# 1. ── CONFIGURATION & WIDGETS ──────────────────────────────────────────────
dbutils.widgets.text("project_catalog", "vstone_catalog")
dbutils.widgets.text("gold_schema",     "gold")
dbutils.widgets.text("silver_schema",   "silver")

CATALOG = dbutils.widgets.get("project_catalog")
GOLD    = f"{CATALOG}.{dbutils.widgets.get('gold_schema')}"
SILVER  = f"{CATALOG}.{dbutils.widgets.get('silver_schema')}"

# 2. ── UTILITY FUNCTION (Fixes NameError: record) ───────────────────────────
ALL_RESULTS = []
def record(suite, name, status, detail=""):
    """Logs test results to a global list and prints to console."""
    ALL_RESULTS.append({"suite": suite, "test": name, "status": status, "detail": detail})
    tag = " PASS" if status == "PASS" else " FAIL"
    print(f"  {tag} | {suite} | {name:<45} | {detail}")

# 3. ── T2: ROW-TO-ROW INTEGRITY ─────────────────────────────────────────────
print(f"\nT2 ── ROW-TO-ROW INTEGRITY: FULL STAR SCHEMA ────────────────────────")
print(f"Checking Source: {SILVER} | Target: {GOLD}")

# --- Fact Table Integrity (1:1 with Silver) ---
silver_fact_df = spark.table(f"{SILVER}.listings_silver_merged").select("listing_id")
gold_fact_df   = spark.table(f"{GOLD}.fact_listings").select("listing_id")

missing_fact  = silver_fact_df.subtract(gold_fact_df).count()
invented_fact = gold_fact_df.subtract(silver_fact_df).count()

record("T2", "fact_listings: All Silver IDs present", 
       "PASS" if missing_fact == 0 else "FAIL", f"Missing={missing_fact:,}")
record("T2", "fact_listings: No invented IDs", 
       "PASS" if invented_fact == 0 else "FAIL", f"Invented={invented_fact:,}")

# --- SCD2 Dimensions Integrity (Active Rows Match) ---
scd2_config = [
    ("dim_car",             f"{SILVER}.car_catalog_transformation",   ["brand", "model"]),
    ("dim_location",        f"{SILVER}.geography_transformation",     ["city_prepositional"]),
    ("dim_listing_details", f"{SILVER}.listings_text_transformation",  ["listing_id"]),
    ("dim_listing_photos",  f"{SILVER}.listings_photo_transformation", ["listing_id", "photo_url_clean"])
]

for dim, silver_tbl, keys in scd2_config:
    # Distinct keys from Silver
    silver_keys = spark.table(silver_tbl).select(*keys).distinct()
    
    # Active records from Gold (DLT SCD2 metadata: __END_AT IS NULL)
    gold_active_keys = spark.table(f"{GOLD}.{dim}").filter("__END_AT IS NULL").select(*keys)

    missing_in_gold  = silver_keys.subtract(gold_active_keys).count()
    invented_in_gold = gold_active_keys.subtract(silver_keys).count()

    record("T2", f"{dim}: Silver keys matched in Gold Active",
           "PASS" if missing_in_gold == 0 else "FAIL", f"Missing={missing_in_gold:,}")
    
    record("T2", f"{dim}: No invented keys in Gold Active",
           "PASS" if invented_in_gold == 0 else "FAIL", f"Invented={invented_in_gold:,}")

## T3 — Audit Columns

In [0]:

print("\nT3 ── AUDIT COLUMNS ──────────────────────────────────────────────────")


all_gold_tables = [
    "dim_date", "dim_car", "dim_location", "dim_listing_details", "dim_listing_photos",
    "fact_listings", "agg_monthly_sales_trend", "agg_brand_location_performance",
    "agg_regional_market_depth", "agg_comprehensive_kpi_cube", "agg_top_10_brands_by_spend"
]

for tbl in all_gold_tables:
    df      = spark.table(f"{GOLD}.{tbl}")
    has_col = "gold_load_dt" in df.columns
    # Check if column exists and has no NULLs
    nulls   = df.filter(F.col("gold_load_dt").isNull()).count() if has_col else -1
    record("T3", f"{tbl} has gold_load_dt NOT NULL",
           "PASS" if has_col and nulls == 0 else "FAIL",
           f"Exists={has_col}  NULL rows={nulls:,}")

# T3.2 SCD2 Metadata verification (DLT Native START/END columns)
# DLT SCD2 tracks history using __START_AT and __END_AT
scd2_dims = ["dim_car", "dim_location", "dim_listing_details", "dim_listing_photos"]

for dim in scd2_dims:
    cols        = spark.table(f"{GOLD}.{dim}").columns
    has_start   = "__START_AT"     in cols
    has_end     = "__END_AT"       in cols
    has_silver  = "silver_load_dt" in cols # Preserved from Silver layer
    
    # Logic: DLT uses __END_AT IS NULL for active records; no __CURRENT needed
    ok = has_start and has_end and has_silver
    record("T3", f"{dim} has SCD2 metadata (__START_AT, __END_AT, silver_load_dt)",
           "PASS" if ok else "FAIL",
           f"START={has_start} END={has_end} silver_load_dt={has_silver}")

# T3.3 Full Lineage Traceability (fact_listings chain)
# Ensure the full audit chain is preserved: Bronze -> Silver -> Gold
fact_cols     = spark.table(f"{GOLD}.fact_listings").columns
audit_chain   = ["bronze_load_dt", "bronze_source_file", "silver_load_dt", "gold_load_dt"]
missing_audit = [c for c in audit_chain if c not in fact_cols]

record("T3", "fact_listings full audit chain (bronze→silver→gold)",
       "PASS" if not missing_audit else "FAIL",
       f"Missing={missing_audit if missing_audit else 'None'}")

## T4 — Schema Integrity

In [0]:
# ══════════════════════════════════════════════════════════════════════════════
# T4 — SCHEMA INTEGRITY (Complete Star Schema Verification)
# ══════════════════════════════════════════════════════════════════════════════
print("\nT4 ── SCHEMA INTEGRITY: FULL STAR SCHEMA ──────────────────────────────")

# 1. FACT_LISTINGS Verification
fact_cols = spark.table(f"{GOLD}.fact_listings").columns
fk_cols        = ["listing_id", "listing_date", "brand", "model", "location_key"]
derived_cols   = ["car_age_at_listing", "is_high_mileage", "price_per_hp_usd", "photo_count"]
financial_cols = ["price_rub", "price_usd", "price_category", "car_age_years", "listing_year", "listing_month"]
audit_cols     = ["bronze_load_dt", "bronze_source_file", "silver_load_dt", "gold_load_dt"]

missing_fact = [c for c in fk_cols + derived_cols + financial_cols + audit_cols if c not in fact_cols]
record("T4", "fact_listings: all FK + derived + financial + audit columns",
       "PASS" if not missing_fact else "FAIL",
       f"Missing={missing_fact if missing_fact else 'None'}  Total={len(fact_cols)}")

# 2. DIM_DATE Verification
date_cols     = spark.table(f"{GOLD}.dim_date").columns
date_required = ["date_key", "year", "quarter", "month", "month_name", "week_of_year", "day", "day_name", "is_weekend", "gold_load_dt"]
missing_date  = [c for c in date_required if c not in date_cols]
record("T4", "dim_date: all calendar + audit columns",
       "PASS" if not missing_date else "FAIL",
       f"Missing={missing_date if missing_date else 'None'}")

## T5 — Star Schema Join Quality

In [0]:

print("\nT5 ── STAR SCHEMA JOIN QUALITY ───────────────────────────────────────")

fact_cnt = spark.table(f"{GOLD}.fact_listings").count()

# Rule: Joining Fact with the 'Active' version of each SCD2 Dimension
joins = [
    ("dim_date",            "f.listing_date = d.date_key",                                              None),
    ("dim_car",             "lower(trim(f.brand)) = lower(trim(d.brand)) AND lower(trim(f.model)) = lower(trim(d.model))", "__END_AT IS NULL"),
    ("dim_location",        "f.location_key = d.city_prepositional",                                    "__END_AT IS NULL"),
    ("dim_listing_details", "f.listing_id = d.listing_id",                                              "__END_AT IS NULL"),
    ("dim_listing_photos",  "f.listing_id = d.listing_id",                                              "__END_AT IS NULL"), # New Dim added
]

for dim, condition, current_filter in joins:
    # Build SQL to join fact and dimension
    where = f"WHERE d.{current_filter}" if current_filter else ""
    sql = f"SELECT COUNT(*) AS c FROM {GOLD}.fact_listings f JOIN {GOLD}.{dim} d ON {condition} {where}"
    
    joined = spark.sql(sql).collect()[0]['c']
    
    # Calculate match rate. For dim_listing_photos, count might exceed fact_cnt due to 1:N 
    # but for join quality we just want to see if links exist
    rate = round(joined / fact_cnt * 100, 2)
    
    record("T5", f"fact → {dim} join rate ≥ {MIN_JOIN_RATE_PCT}%",
           "PASS" if rate >= MIN_JOIN_RATE_PCT else "FAIL",
           f"Matched={joined:,}/{fact_cnt:,}  Rate={rate}%")



## T6 — SCD2 Integrity

In [0]:

print("\nT6 ── SCD2 INTEGRITY ─────────────────────────────────────────────────")

scd2_config = [
    ("dim_car",             ["brand","model"]),
    ("dim_location",        ["city_prepositional"]),
    ("dim_listing_details", ["listing_id"]),
]

for dim, keys in scd2_config:
    key_str = ", ".join(keys)

    # 1. Exactly one Active Record per key
    # DLT uses __END_AT IS NULL to identify the current record.
    dup_sql = (f"SELECT COUNT(*) AS c FROM ("
               f"SELECT {key_str}, COUNT(*) AS cnt FROM {GOLD}.{dim} "
               f"WHERE __END_AT IS NULL GROUP BY {key_str} HAVING cnt > 1)")
    
    dup = spark.sql(dup_sql).collect()[0]['c']
    record("T6", f"{dim}: exactly one active record (__END_AT IS NULL) per key",
           "PASS" if dup==0 else "FAIL", f"Keys with multiple active rows={dup:,}")

    # 2. Timeline Consistency Check
    # Ensure start date is always before end date for historical records.
    timeline_sql = (f"SELECT COUNT(*) AS c FROM {GOLD}.{dim} "
                    f"WHERE __END_AT IS NOT NULL AND __START_AT >= __END_AT")
    
    bad_timeline = spark.sql(timeline_sql).collect()[0]['c']
    record("T6", f"{dim}: __START_AT is before __END_AT for history",
           "PASS" if bad_timeline==0 else "FAIL", f"Timeline violations={bad_timeline:,}")

    # 3. Null Termination Check
    # Every natural key must have exactly one row that is currently active.
    missing_active_sql = (f"SELECT COUNT(*) AS c FROM ("
                          f"SELECT {key_str} FROM {GOLD}.{dim} "
                          f"GROUP BY {key_str} HAVING SUM(CAST(__END_AT IS NULL AS INT)) = 0)")
    
    missing_active = spark.sql(missing_active_sql).collect()[0]['c']
    record("T6", f"{dim}: every key has at least one active row",
           "PASS" if missing_active==0 else "FAIL", f"Keys with no active row={missing_active:,}")